# repair2 데이터 분석

- 원본 데이터: 같은 폴더의 `repair2.csv`
- 현재 작업 범위: 결측값 확인 및 데이터 타입 변환(datetime 포함)
- 진행 순서: 결측값 확인 → 데이터 타입 변환 → 변환 결과 점검 → 결측값 처리

## 1. 데이터 불러오기 및 결측값 확인

원본 데이터를 불러오고 행·열 수, 칼럼별 자료형과 결측값 확인
원본을 유지하기 위해 DataFrame을 복사한 뒤 작업

In [56]:
import pandas as pd

raw = pd.read_csv('repair2.csv', encoding='utf-8-sig')

df = raw.copy(deep=True) #원본을 유지하기 위해 copy()를 사용하여 DataFrame을 복사합니다.

df.head()

,caseID,taskID,originator,eventtype,contact,RepairType,objectKey,RepairInternally,EstimatedRepairTime,RepairCode,RepairOK,date,time
0,419,InformClientSurvey,System,complete,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-06,11:51
1,561,ArrangeSurvey,Monica,complete,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-07,13:48
2,902,ReadyInformClient,System,complete,NaN,NaN,NaN,NaN,NaN,NaN,True,2020-01-08,16:20
3,18,Survey,Edd,start,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2020-01-09,6:55
4,852,Survey,Edd,complete,NaN,NaN,NaN,True,240.0,1.0,NaN,2020-01-10,11:33


In [57]:
print(df.info())
print("===="*20)

print(f"Shape of the DataFrame: {df.shape}")
print("===="*20)

# 각 컬럼의 결측값 확인
df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13262 entries, 0 to 13261
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   caseID               13262 non-null  int64  
 1   taskID               13260 non-null  object 
 2   originator           13260 non-null  object 
 3   eventtype            13257 non-null  object 
 4   contact              1000 non-null   object 
 5   RepairType           927 non-null    object 
 6   objectKey            927 non-null    float64
 7   RepairInternally     927 non-null    object 
 8   EstimatedRepairTime  927 non-null    float64
 9   RepairCode           927 non-null    float64
 10  RepairOK             1854 non-null   object 
 11  date                 13249 non-null  object 
 12  time                 13249 non-null  object 
dtypes: float64(3), int64(1), object(9)
memory usage: 1.3+ MB
None
Shape of the DataFrame: (13262, 13)


caseID                     0
taskID                     2
originator                 2
eventtype                  5
contact                12262
RepairType             12335
objectKey              12335
RepairInternally       12335
EstimatedRepairTime    12335
RepairCode             12335
RepairOK               11408
date                      13
time                      13
dtype: int64

**관찰·판단 기록**

- 확인한 사실: date와 time 칼럼의 type 수정 필요
- 처리 기준과 이유: 원본을 유지하기 위해 copy()를 사용하여 DataFrame을 복사
- 다음에 확인할 것:

## 2. 데이터 타입 변환

EstimatedRepairTime의 숫자형 확인 및 RepairInternally·RepairOK의 boolean 변환
변환 전후 자료형과 결측값 수를 비교하고 기존 값이 유지되는지 확인

In [63]:
df['EstimatedRepairTime'] = pd.to_numeric(
    df['EstimatedRepairTime'], errors='coerce')
print("===="*20)

# 실제 어떤 값이 있는지 확인
print(df['RepairInternally'].value_counts(dropna=False))
print(df['RepairOK'].value_counts(dropna=False))
print("===="*20)

# # 문자 'TRUE'/'FALSE'를 boolean 타입으로 변환
df['RepairInternally'] = df['RepairInternally'].astype('boolean')
df['RepairOK'] = df['RepairOK'].astype('boolean')

# print(df.dtypes)
df.isna().sum()

RepairInternally
<NA>     12335
True       868
False       59
Name: count, dtype: Int64
RepairOK
<NA>    11408
True     1854
Name: count, dtype: Int64


caseID                     0
taskID                     2
originator                 2
eventtype                  5
contact                12262
RepairType             12335
objectKey              12335
RepairInternally       12335
EstimatedRepairTime    12335
RepairCode             12335
RepairOK               11408
date                      13
time                      13
timestamp                 13
dtype: int64

**관찰·판단 기록**

- 확인한 사실: 다수의 결측치를 가진 칼럼이 존재
- 처리 기준과 이유: 결측치가 필요에 의한 결측치인지 무의미한 결측치인지 확인 필요
- 다음에 확인할 것: 

## 3. datetime 변환

date와 time을 결합해 timestamp 생성
원본 칼럼은 유지하고 원본 시각 누락과 형식 변환 실패를 구분

In [ ]:
# 날짜와 시간을 합쳐 datetime으로 변환
df['timestamp'] = pd.to_datetime(
    df['date'] + ' ' + df['time'],
    format='%Y-%m-%d %H:%M', errors='coerce')

# 원본부터 비어있던 행
missing_source = df['date'].isna() | df['time'].isna()

print('원본 시각 누락:', missing_source.sum())
print('형식 변환 실패:', parse_failure.sum())
print(df.loc[df['timestamp'].isna(), ['caseID', 'date', 'time']])

원본 시각 누락: 13
형식 변환 실패: 0
       caseID date time
860       992  NaN  NaN
1078      113  NaN  NaN
3135      129  NaN  NaN
4526      100  NaN  NaN
5771      119  NaN  NaN
6225      123  NaN  NaN
6612      180  NaN  NaN
8144      146  NaN  NaN
9715      116  NaN  NaN
10235     154  NaN  NaN
10369     101  NaN  NaN
10406     106  NaN  NaN
10526     154  NaN  NaN


**관찰·판단 기록**

- 확인한 사실
    - 전체 13,262행 중 날짜·시간 결측값이 13행에서 확인됨. 총 12개 케이스에 해당함
    - 결측값은 start 4건, complete 8건, eventtype 미기록 1건으로 구성됨
    - 원본 결측값 외에 datetime 형식 변환에 실패한 데이터는 없음
- 처리 기준과 이유
    - 현재 단계에서는 결측값을 삭제하거나 대체하지 않고 유지함
    - 시간 정보가 없더라도 업무 수행 기록은 프로세스 흐름 분석에 활용할 수 있기 때문임
    - 소요 시간 분석 시 결측값이 계산 결과에 미치는 영향을 확인한 후 제외 여부를 결정할 예정임
- 다음에 확인할 것
    - 결측값이 발생한 업무와 케이스의 특성 확인
    - 업무별 start,complete 이벤트의 정상적인 연결 여부 확인
    - 결측값 및 날짜 이상값이 소요 시간과 병목 분석에 미치는 영향 검토
    - 처음부터 재실행 시 `parse_failure` 변수를 출력하기 전 정의 여부 확인

## 4. 변환 결과 점검

최초 결측값과 타입 변환 후 결측값 비교
연도 분포를 확인하고 날짜 이상값의 추가 검토 필요 여부 확인

In [ ]:
print('처음 불러왔을 때의 결측값')
print(raw.isna().sum())
print("===="*20)

print('타입 변환 후의 결측값')
print(df.isna().sum())
print("===="*20)

print('변환된 날짜의 연도 분포')
print(df['timestamp'].dt.year.value_counts(dropna=False))

처음 불러왔을 때의 결측값
caseID                     0
taskID                     2
originator                 2
eventtype                  5
contact                12262
RepairType             12335
objectKey              12335
RepairInternally       12335
EstimatedRepairTime    12335
RepairCode             12335
RepairOK               11408
date                      13
time                      13
dtype: int64
타입 변환 후의 결측값
caseID                     0
taskID                     2
originator                 2
eventtype                  5
contact                12262
RepairType             12335
objectKey              12335
RepairInternally       12335
EstimatedRepairTime    12335
RepairCode             12335
RepairOK               11408
date                      13
time                      13
timestamp                 13
dtype: int64
변환된 날짜의 연도 분포


timestamp
1970.0    12383
2021.0      365
2020.0      361
2022.0      134
NaN          13
1905.0        6
Name: count, dtype: int64

**관찰·판단 기록**

- 확인한 사실:
- 처리 기준과 이유:
- 다음에 확인할 것:

## 5. 결측값 처리

결측값 확인과 타입 변환을 마친 뒤 분석 목적에 따라 유지·보류·보완 기준 결정
처리 전후 행 수와 케이스 수를 비교하고 판단 근거 기록

**관찰·판단 기록**

- 확인한 사실:
- 처리 기준과 이유:
- 다음에 확인할 것:

## 6. 정렬·저장

이벤트를 정렬하고 전처리 결과를 원본과 분리해 저장
보류한 데이터와 처리 기준을 함께 기록

**관찰·판단 기록**

- 확인한 사실:
- 처리 기준과 이유:
- 다음에 확인할 것:

## 7. 프로세스 분석

분석 지표와 시작·종료 기준을 정하고 주요 프로세스 흐름 확인
이후 병목·반복작업, 소요 시간 관련 요인과 개선안 검토

**관찰·판단 기록**

- 확인한 사실:
- 처리 기준과 이유:
- 다음에 확인할 것: